In [1]:
import numpy as np

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [3]:
def V_shaped_function(x):
    return np.abs(x)/1+np.abs(x)

In [4]:
class Particle: 
    swarm_best_position = None
    swarm_best_fitness = float('-inf')
    
    iter_count = 0
    
    def __init__(self, X_train, y_train, calc_accuracy_fn, alpha, num_iters, c_inertia_start, c_inertia_end, c_social, c_cognitive):
        self.X_train = X_train
        self.y_train = y_train
        self.calc_accuracy_fn = calc_accuracy_fn
        self.alpha = alpha
        self.num_iters = num_iters
        
        self.c_inertia_start = c_inertia_start
        self.c_inertia_end = c_inertia_end
        self.c_social = c_social
        self.c_cognitive = c_cognitive
        self.position = self.initialize_position()
        self.velocity = self.initialize_velocity(self.X_train.shape[1])

        self.fitness = self.calc_fitness()
        self.personal_best_position = self.position.copy()
        self.personal_best_fitness = self.fitness

        if Particle.swarm_best_position is None or Particle.swarm_best_fitness < self.fitness:
            Particle.swarm_best_fitness = self.fitness
            Particle.swarm_best_position = self.position.copy()
        
    def calc_fitness(self): 
        if not any(self.position):
            return float('-inf')
        acc = self.calc_accuracy_fn(self.position, self.X_train, self.y_train)
        num_features = sum(self.position)
        return self.alpha * acc + (1 - self.alpha) * (1 - num_features / self.X_train.shape[1])
    
    def initialize_position(self):
        return np.random.choice([True,False], size = self.X_train.shape[1])
    
    def initialize_velocity(self, num_dimensions):
        return np.random.uniform(-1, 1, size=num_dimensions)
    
    def update_velocity(self):
        r_s = np.random.random(len(self.position))
        r_c = np.random.random(len(self.position))
        assert self.swarm_best_position is not None
        
        self.c_inertia = self.c_inertia_start - (self.c_inertia_start - self.c_inertia_end)*(Particle.iter_count/self.num_iters)
        social_velocity = self.c_social * r_s * ((self.swarm_best_position.astype(float)) - self.position.astype(float))
        cognitive_velocity = self.c_cognitive * r_c * ((self.swarm_best_position.astype(float)) - self.position.astype(float))
        
        self.velocity = self.c_inertia * self.velocity + social_velocity + cognitive_velocity
        self.velocity = np.clip(self.velocity, -4, 4)
        
    def move(self):
        Particle.iter_count += 1;
        self.update_velocity();
        probabilities = V_shaped_function(self.velocity)
        random_vector = np.random.rand(len(probabilities))
        flip = random_vector < probabilities
        self.position[flip] = ~self.position[flip]
        
        self.fitness = self.calc_fitness()
        if self.fitness > self.personal_best_fitness:
            self.personal_best_position = self.position.copy()
            self.personal_best_fitness = self.fitness
            if self.fitness > Particle.swarm_best_fitness:
                Particle.swarm_best_position = self.position.copy()
                Particle.swarm_best_fitness = self.fitness

In [5]:
def BPSO(X_train, y_train, calc_accuracy_fn, alpha, num_iters, swarm_size, c_inertia_start, c_inertia_end, c_social, c_cognitive):
    Particle.swarm_best_position = None
    Particle.swarm_best_fitness = float('-inf')
    swarm = [Particle(X_train, y_train, calc_accuracy_fn, alpha, num_iters, c_inertia_start, c_inertia_end, c_social, c_cognitive) for _ in range(swarm_size)]
    for _ in range(num_iters):
        for p in swarm:
            p.move()
    return Particle.swarm_best_position, Particle.swarm_best_fitness

In [6]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer

In [7]:
def evaluate_knn(code,X_train,y_train,n_splits = 5,scoring = 'accuracy', k = 5):    
    selected_features = X_train.columns[code]    
    model = KNeighborsClassifier(k)
    scores = cross_val_score(model, X_train[selected_features], y_train, cv=n_splits, scoring='accuracy') 
    return scores.mean()

In [8]:
X_train = pd.read_csv("../datasets/ionosphere/X_train.csv")
y_train = pd.read_csv('../datasets/ionosphere/y_train.csv').to_numpy().ravel()
X_test = pd.read_csv("../datasets/ionosphere/X_test.csv")
y_test = pd.read_csv("../datasets/ionosphere/y_test.csv").to_numpy().ravel()

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

In [11]:
X = df.drop("target",axis=1)
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [12]:
y = df[["target"]]
y.head()

,target
0,0
1,0
2,0
3,0
4,0


In [15]:
X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(X,y,test_size=0.3)
y_train_cancer = y_train_cancer.to_numpy().ravel()

In [16]:
solution, fitness = BPSO(
    X_train=X_train_cancer,
    y_train=y_train_cancer,
    calc_accuracy_fn=evaluate_knn,
    alpha=0.5,
    num_iters=100,
    swarm_size=10,
    c_inertia_start=0.9,
    c_inertia_end=0.4,
    c_social=2.0,
    c_cognitive=1.5
)

In [15]:
fitness

0.9094883966244727

In [16]:
sum(solution)/len(solution)

0.13333333333333333

In [17]:
sum(solution)

4

In [18]:
len(solution)

30